# Hard-coding a single iteration with sparse operations
We will develop cupy code for a single iteration of our sparse tensor factorization model.
We build upon the previous notebook where we developed a first dot product with sparse operations (which is the main bottleneck of the algorithm).

In [1]:
# we reload the packages as we make changes frequently
%load_ext autoreload
%autoreload 2

In [2]:
from tensorly_custom.contrib.sparse.decomposition import non_negative_tucker
import tensorly_custom as tl
import numpy as np
import pickle
import torch
import os
from typing import List, Tuple
from utils import DATA_DIR, select_gpu, tree_to_device
from typing import Optional, Union
import cupy as cp
import cupyx.scipy.sparse as cpx_sparse
import sparse
import tensorflow as tf

2025-11-28 13:46:53.735256: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
device = select_gpu(1)
def _to_np(x):
    # Accept NumPy arrays or torch tensors; return NumPy view/copy
    if hasattr(x, "detach"):  # torch.Tensor
        return x.detach().cpu().numpy()
    return x

def _torch_or_tucker_load(path, map_location="cpu"):
    """Tries to load a torch-saved file, if fails, tries pickle."""
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except RuntimeError:
        with open(path, "rb") as f:
            return pickle.load(f)

def cupy_to_torch_sparse(
    cu_mat: cpx_sparse.spmatrix,
    orig_shape: Optional[Tuple[int, ...]] = None,
    device: Union[str, torch.device] = "cpu",
    dtype: Optional[torch.dtype] = None,
) -> torch.Tensor:
    """
    Convert a CuPy sparse matrix (any format) back to a torch sparse COO tensor.

    If orig_shape is None:
        - The torch tensor is 2D and has the same shape as cu_mat.
    If orig_shape is provided and len(orig_shape) == 2:
        - The torch tensor is 2D with that shape.
    If orig_shape is provided and len(orig_shape) > 2:
        - We treat cu_mat.row as the flattened N-D index and unflatten it
          back to N-D using np.unravel_index, assuming the representation
          created by `torch_sparse_to_cupy`.

    Args:
        cu_mat: CuPy sparse matrix (COO/CSR/CSC, will be converted to COO).
        orig_shape: original tensor shape (for N-D tensors).
        device: target torch device.
        dtype: target dtype for values (defaults to inferred from data).

    Returns:
        torch.sparse_coo_tensor on the requested device.
    """

    if not cpx_sparse.isspmatrix_coo(cu_mat):
        cu_mat = cu_mat.tocoo()

    row_cp = cu_mat.row
    col_cp = cu_mat.col
    data_cp = cu_mat.data

    row_np = cp.asnumpy(row_cp)
    col_np = cp.asnumpy(col_cp)
    data_np = cp.asnumpy(data_cp)

    if orig_shape is None:
        shape = cu_mat.shape
        indices_np = np.vstack([row_np, col_np])
    else:
        shape = tuple(orig_shape)
        if len(shape) == 2:
            indices_np = np.vstack([row_np, col_np])
        else:
            # --- decode block encoding ---
            size = int(np.prod(shape))
            int32_max = np.iinfo(np.int32).max
            block_size = min(size, int32_max)

            flat = row_np + col_np * block_size
            coords = np.unravel_index(flat, shape)
            indices_np = np.vstack(coords)

    indices_t = torch.from_numpy(indices_np).long()
    values_t = torch.from_numpy(data_np)
    if dtype is not None:
        values_t = values_t.to(dtype)

    x = torch.sparse_coo_tensor(indices_t, values_t, size=shape)
    x = x.coalesce().to(device)
    return x

def torch_sparse_to_cupy(
    x: torch.Tensor,
) -> Tuple[cpx_sparse.coo_matrix, Tuple[int, ...]]:
    if not x.is_sparse:
        raise TypeError("torch_sparse_to_cupy expects a torch sparse tensor (COO).")

    x = x.coalesce()
    indices = x.indices()   # (ndim, nnz)
    values = x.values()     # (nnz,)
    shape = tuple(x.shape)

    indices_np = indices.cpu().numpy()
    values_np = values.cpu().numpy()

    ndim, nnz = indices_np.shape

    if ndim == 2:
        # unchanged
        row = indices_np[0]
        col = indices_np[1]
        row_cp = cp.asarray(row)
        col_cp = cp.asarray(col)
        data_cp = cp.asarray(values_np)
        cu_mat = cpx_sparse.coo_matrix((data_cp, (row_cp, col_cp)), shape=shape)
    else:
        # --- NEW BLOCK ENCODING ---
        coords = [indices_np[d] for d in range(ndim)]
        size = int(np.prod(shape))  # prod of original N-D shape
        flat = np.ravel_multi_index(coords, shape)  # 0..size-1

        int32_max = np.iinfo(np.int32).max
        block_size = min(size, int32_max)
        # number of blocks on the column axis
        n_blocks = (size + block_size - 1) // block_size

        row = flat % block_size
        col = flat // block_size

        row_cp = cp.asarray(row, dtype=cp.int32)
        col_cp = cp.asarray(col, dtype=cp.int32)
        data_cp = cp.asarray(values_np)

        cu_mat = cpx_sparse.coo_matrix(
            (data_cp, (row_cp, col_cp)),
            shape=(block_size, n_blocks),
        )

    return cu_mat, shape



def _role_index(role: str) -> int:
    if role == "verb":
        return 0
    elif role == "subject":
        return 1
    elif role == "object":
        return 2
    else:
        raise ValueError("role must be one of {'verb','subject','object'}")


class SparseTupleTensor:
    """Encapsulating the Sparse TupleTensor (built from vectors extracted from corpus) and the vocabulary,
    providing methods for decomposition, refactoring, etc.."""
    def __init__(self, tensor, device="cpu", sparsity_type=None):
        self.tensor = tensor
        self.sparsity_type = sparsity_type
        self.shape = tensor.shape
        self.device = device

    # --- Construction and loading ---
    @classmethod
    def load_from_disk(cls,
                       dataset: str="karrewiet_sparse",
                       method: str="counting",
                       dims: int=1000,
                       map_location: str="cpu",
                       sparsity_type: Optional[str]=None
                          ) -> "SparseTupleTensor":

        """Loads a precomputed tucker decomposition from disk.
            Args:
                dataset (str): name of the dataset
                method (str): method used to compute the decomposition
                    - one of "counting", "sc", "sii"
                dims (int): dimensionality of the original tensor modes (vocab size)
                rank (int): rank of the decomposition
                iterations (int): number of iterations used to compute the decomposition
                map_location (str): device to map the loaded tensors to
            Returns:
                ((core, factors), vocab)
                    core: torch.Tensor
                    factors: list[torch.Tensor]
                    vocab: dict with keys 'vocab_v','vocab_s','vocab_o','v2i','s2i','o2i'
        """
        if method not in {"counting", "sc", "sii"}:
            raise ValueError("method must be one of {'counting','sc','sii'}")
        base = os.path.join(DATA_DIR, "tensors", dataset)

        # vocab_path = os.path.join(base, f"vocabularies/{dims}.pkl")
        populated_path = os.path.join(base,"populated", f"{method}_{dims}.pt")
        # if not os.path.exists(vocab_path):
        #     raise FileNotFoundError(f"Missing vocab file: {vocab_path}")
        if not os.path.exists(populated_path):
            raise FileNotFoundError(f"Missing decomposition file: {populated_path}")
        # the vocab is here under f"vocabularies_[dims].pkl"
        # Load with torch (they were saved with torch.save)
        # with open(vocab_path, "rb") as f:
        #     vocab = pickle.load(f)
        tensor = _torch_or_tucker_load(populated_path, map_location=map_location)

        return cls(tensor, device=map_location, sparsity_type=sparsity_type)



    # -- Sparsity methods ---
    def sparse_representation(self, sparse_type):
        # we return the sparse representation of the tensor
        if sparse_type == self.sparsity_type:
            return self.tensor
        # we check if our tensor is a tensorflow tensor or make it one
        if sparse_type == "tensorflow":
            if self.sparsity_type != "torch":
                tensor = self.sparse_representation("torch")
            else:
                tensor = self.tensor
            # we build from torch sparse tensor
            indices = tensor.coalesce().indices().t().numpy()   # shape (nnz, ndim)
            values  = tensor.coalesce().values().numpy()        # shape (nnz,)
            shape   = tuple(self.shape)          # e.g. (d0, d1, ..., d_{n-1})
            sparse_tensor = tf.SparseTensor(indices=indices, values=values, dense_shape=shape)
            # we warn users that tensorflow sparse tensors map directly to GPU.
            # additionally, they directly "allocate" the whole GPU memory to tf to reduce fragmentation later on.
            # this makes nvtop commands etc. not useable anymore

            print("WARNING: TensorFlow sparse tensors are allocated on GPU and may reserve large amounts of GPU memory.")

            return sparse_tensor

        elif sparse_type == "torch":
            if not self.sparsity_type or self.sparsity_type == "dense":
               return self.tensor.to_sparse()
            # can work from any tensor-like object
            elif self.sparsity_type == "cupy":
                return cupy_to_torch_sparse(self.tensor, orig_shape=self.shape)
            elif self.sparsity_type == "tensorflow":
                coords = self.tensor.indices.numpy()       # shape (nnz, ndim)
                data   = self.tensor.values.numpy()        # shape (nnz,)
                shape  = tuple(self.shape)  # e.g. (d0, d1, ..., d_{n-1})
                sparse_tensor = torch.sparse_coo_tensor(torch.tensor(coords).t(), torch.tensor(data), size=shape, device="cpu")
                return sparse_tensor
            else:
                raise NotImplementedError("sparsity_type must be one of {'dense', None, 'cupy', 'tensorflow','torch'}")

        elif sparse_type == "sparse":
            # can only work from a sparse torch tensor
            if not isinstance(self.tensor, torch.Tensor) or not self.tensor.is_sparse:
                raise TypeError("sparse expects self.tensor to be a torch sparse tensor.")
            coords = self.tensor.indices().numpy()       # shape (nnz, ndim)
            data   = self.tensor.values().numpy()        # shape (nnz,)
            shape  = tuple(self.tensor.size())  # e.g. (d0, d1, ..., d_{n-1})
            sparse_tensor = sparse.COO(coords, data, shape=shape)
            return sparse_tensor

        elif sparse_type == "cupy":
            if not isinstance(self.tensor, torch.Tensor) or not self.tensor.is_sparse:
                raise TypeError("cupy expects self.tensor to be a torch sparse tensor.")
            tensor_cupy, shape = torch_sparse_to_cupy(self.tensor)
            return tensor_cupy
        else:
            raise NotImplementedError(f"Sparse representation for type {sparse_type} not implemented.")



    def tensor_to_sparse(self, sparse_type="tensorflow"):
        self.tensor = self.sparse_representation(sparse_type)
        self.sparsity_type = sparse_type
        if sparse_type in ["tensorflow", "cupy"]:
            self.device = "cuda"


    def tensor_to_dense(self):
        if not isinstance(self.tensor, torch.Tensor) or not self.tensor.is_sparse:
            raise TypeError("tensor_to_dense expects self.tensor to be a torch sparse tensor.")
        self.tensor = self.tensor.to_dense()
        self.sparsity_type = "dense"

    def to_device(self, device):
        self.tensor = tree_to_device(self.tensor, device)
        self.device = device
        if device == "cpu":
            torch.cuda.empty_cache()

    def inspect(self):
        print("type:", type(self.tensor))
        print("sparsity type:", self.sparsity_type)
        print("shape:", self.shape)
        print("device:", self.device)

        if not self.sparsity_type or self.sparsity_type == "dense":
            memory_size = self.tensor.element_size() * self.tensor.nelement()
        elif self.sparsity_type == "torch":
            nnz = self.tensor._nnz()
            dtype_size = self.tensor.values().element_size()
            memory_size = nnz * (self.tensor.indices().element_size() * self.tensor.indices().shape[0] + dtype_size)

        elif self.sparsity_type == "cupy":
            memory_size = self.tensor.data.nbytes + self.tensor.row.nbytes + self.tensor.col.nbytes
        elif self.sparsity_type == "sparse":
            memory_size = self.tensor.nbytes
        elif self.sparsity_type == "tensorflow":
            nnz = self.tensor.values.shape[0]
            dtype_size = self.tensor.values.dtype.size
            memory_size = nnz * (self.tensor.indices.dtype.size * self.tensor.indices.shape[1] + dtype_size)

        else:
            memory_size = self.tensor.nbytes
        print(f"approx. memory size: {memory_size / (1024**2):.2f} MB")



In [4]:
def left_dense_mul_sparse(
    mat: cp.ndarray,
    sp: cpx_sparse.spmatrix,
    force_sparse: bool = False,
    max_dense_bytes: int = 2**31,  # ~2 GB default safety cap
    use_dense: bool = False,
) -> Union[cp.ndarray, cpx_sparse.coo_matrix]:
    """
    Compute mat @ sp, choosing dense or sparse output based on a simple
    memory heuristic.

    mat: cupy ndarray of shape (R, I_mode)
    sp:  cupy sparse matrix of shape (I_mode, K)
    """
    sp = sp.tocoo()
    R, I_mode = mat.shape
    assert I_mode == sp.shape[0], f"mat shape {mat.shape} not compatible with sparse {sp.shape}"

    # Let CuPy handle dense @ sparse; result is cupy.ndarray
    return mat @ sp
def fold_unfolded_sparse_to_vec(
    unfolded: Union[cpx_sparse.spmatrix, cp.ndarray],
    old_shape: Tuple[int, ...],
    mode: int,
    new_dim: int,
) -> Tuple[cpx_sparse.coo_matrix, Tuple[int, ...]]:
    """
    Fold a mode-`mode` unfolded matrix back to a vectorized sparse tensor.

    unfolded:
        - sparse COO or any cupyx.scipy.sparse.spmatrix of shape (new_dim, prod(other_dims)), or
        - dense cupy.ndarray of the same shape.
    old_shape : original N-D shape BEFORE replacing dimension at `mode`
    mode      : mode index that was unfolded
    new_dim   : new size at `mode` (typically rank[mode])

    Returns
    -------
    vec_sparse : COO of shape (prod(new_shape), 1)
    new_shape  : tuple of ints, updated tensor shape
    """

    old_shape = tuple(old_shape)
    N = len(old_shape)

    new_shape = list(old_shape)
    new_shape[mode] = new_dim
    new_shape = tuple(new_shape)

    other_shape = tuple(s for i, s in enumerate(old_shape) if i != mode)

    if cpx_sparse.isspmatrix(unfolded):
        unfolded = unfolded.tocoo()
        row = unfolded.row
        col = unfolded.col
        data = unfolded.data
    else:
        row, col = cp.nonzero(unfolded)
        data = unfolded[row, col]

    coords_other = cp.unravel_index(col, other_shape)

    coords_full = []
    idx_other = 0
    for i in range(N):
        if i == mode:
            coords_full.append(row)
        else:
            coords_full.append(coords_other[idx_other])
            idx_other += 1

    coords_full = tuple(coords_full)

    size = int(np.prod(new_shape))
    int32_max = np.iinfo(np.int32).max
    block_size = min(size, int32_max)

    flat = cp.ravel_multi_index(coords_full, new_shape)

    # --- block encoding of flat indices ---
    row_vec = flat % block_size
    col_vec = flat // block_size

    n_blocks = int((size + block_size - 1) // block_size)
    vec_sparse = cpx_sparse.coo_matrix(
        (data, (row_vec, col_vec)),
        shape=(block_size, n_blocks),
    )
    vec_sparse.sum_duplicates()

    return vec_sparse, new_shape

def sparse_mode_dot_vec(
    vec_tensor: cpx_sparse.spmatrix,
    curr_shape: Tuple[int, ...],
    factor: cp.ndarray,
    mode: int,
    transpose_factor: bool = True,
) -> Tuple[cpx_sparse.coo_matrix, Tuple[int, ...]]:
    """
    Perform a mode-`mode` product on a vectorized sparse tensor (prod(curr_shape), 1),
    using a dense factor matrix, and return the new vectorized sparse tensor.

    vec_tensor: sparse COO (prod(curr_shape), 1)
    curr_shape: current tensor shape
    factor:     dense matrix of shape (I_mode, R_mode) (or R_mode, I_mode if transpose_factor=False)
    mode:       mode index in [0, len(curr_shape))
    transpose_factor: if True, use factor.T (for Tucker-style X ×_n W_n^T)

    Returns
    -------
    new_vec:   sparse COO (prod(new_shape), 1)
    new_shape: updated shape, with dimension at `mode` replaced by R_mode
    """
    curr_shape = tuple(curr_shape)
    I_mode = curr_shape[mode]

    # Factor handling
    if transpose_factor:
        # factor is (I_mode, R_mode) => mat is (R_mode, I_mode)
        assert factor.shape[0] == I_mode, f"factor shape {factor.shape} not compatible with dim {I_mode}"
        mat = tl.transpose(factor)  # (R_mode, I_mode)
    else:
        # factor is already (R_mode, I_mode)
        assert factor.shape[1] == I_mode, f"factor shape {factor.shape} not compatible with dim {I_mode}"
        mat = factor

    R_mode = mat.shape[0]

    # 1) Unfold current sparse tensor along this mode (sparse COO)
    unfolded = unfold_from_vectorized_sparse(
        vec_tensor,
        curr_shape,
        mode,
        to_dense=False,
    )  # shape: (I_mode, prod(other_dims))

    # 2) Left-multiply with dense matrix; currently returns dense cp.ndarray
    #    -> shape: (R_mode, prod(other_dims))
    unfolded_new = left_dense_mul_sparse(mat, unfolded)

    # 3) Fold back into a new vectorized sparse tensor with updated shape
    new_vec, new_shape = fold_unfolded_sparse_to_vec(
        unfolded_new,
        old_shape=curr_shape,
        mode=mode,
        new_dim=R_mode,
    )
    return new_vec, new_shape
def sparse_multi_mode_dot_vec(
    vec_tensor: cpx_sparse.spmatrix,
    orig_shape: Tuple[int, ...],
    factors: List[cp.ndarray],
    modes: Optional[List[int]] = None,
    transpose_factors: bool = True,
) -> cp.ndarray:
    """
    multi_mode_dot for a vectorized sparse tensor (prod(orig_shape), 1),
    applying dense factor matrices along the given modes, **staying sparse**
    until the final (small) result, which is densified.

    vec_tensor: sparse COO (prod(orig_shape), 1)
    orig_shape: original tensor shape
    factors:    list of factor matrices, one per mode index
                factor[n] has shape (I_n, R_n)
    modes:      list of modes to apply; if None, uses range(len(factors))
    transpose_factors: if True, uses factors[n].T (Tucker-style)
    """
    if modes is None:
        modes = list(range(len(factors)))

    current_vec = vec_tensor
    current_shape = tuple(orig_shape)

    # Apply each mode in any order (commutes)
    for mode in modes:
        print("Applying mode", mode)
        current_vec, current_shape = sparse_mode_dot_vec(
            current_vec,
            current_shape,
            factors[mode],
            mode=mode,
            transpose_factor=transpose_factors,
        )

    # At this point, current_vec is still sparse (prod(core_shape), 1)
    core_shape = current_shape  # typically your (50, 50, 50) or similar
    print("Final core shape (sparse):", core_shape)
    # Finally densify the small core
    coo = current_vec.tocoo()
    flat = coo.row
    data = coo.data

    # Build dense core
    coords = cp.unravel_index(flat, core_shape)
    core_dense = cp.zeros(core_shape, dtype=data.dtype)
    core_dense[coords] = data

    return core_dense




In [5]:
sparse_tensor = SparseTupleTensor.load_from_disk(dataset="fineweb_sparse",
                                          method="counting",
                                          dims=5000,
                                          map_location="cpu",
                                          sparsity_type="torch")
print("Initial tensor:")
sparse_tensor.inspect()
print("\nConverting to cupy sparse representation:")
sparse_tensor.tensor_to_sparse("cupy")
sparse_tensor.inspect()

Initial tensor:
type: <class 'torch.Tensor'>
sparsity type: torch
shape: torch.Size([5000, 5000, 5000])
device: cpu
approx. memory size: 12.33 MB

Converting to cupy sparse representation:
type: <class 'cupyx.scipy.sparse._coo.coo_matrix'>
sparsity type: cupy
shape: torch.Size([5000, 5000, 5000])
device: cuda
approx. memory size: 5.29 MB


## 0. Initializing rank and epsilon (can be taken over from cupy implementation)


In [6]:
import tensorly_custom as tl
from tensorly_custom.tucker_tensor import validate_tucker_rank
tl.set_backend("cupy")
# we import "validate tucker rank"
shape = tuple(sparse_tensor.shape)
tensor = sparse_tensor.tensor
rank = validate_tucker_rank(shape, rank=(50, 50, 50))
epsilon = 1e-12
print("Using rank:", rank)
modes = list(range(len(rank)))
print("Using modes:", modes)
non_negative = True


Using rank: (50, 50, 50)
Using modes: [0, 1, 2]


## 1. Intializing core and factors

In [7]:
init = "random"
random_state = 42


if init == "random":
    rng = tl.check_random_state(random_state)
    core = tl.tensor(
        rng.random_sample([rank[index] for index in range(len(modes))]) + 0.01,
        **tl.context(tensor),
    )  # Check this
    factors = [
        tl.tensor(
            rng.random_sample((shape[mode], rank[index])), # we changed this to original shape
            **tl.context(tensor),
        )
        for index, mode in enumerate(modes)
    ]

else:
    (core, factors) = init

if non_negative is True:
    factors = [tl.abs(f) for f in factors]
    core = tl.abs(core)

In [8]:
def fro_norm_coo(x):
    # x: cupyx.scipy.sparse.coo_matrix
    data = x.data
    return cp.sqrt((cp.abs(data) ** 2).sum())
tensor_coo = tensor.tocoo()
norm_tensor = cp.sqrt((cp.abs(tensor_coo.data) ** 2).sum())
rec_errors = []

## 2. Iteration
### 2.1 mode iteration
We do the steps for a single mode

In [9]:
from tensorly_custom.decomposition._tucker import tucker_to_tensor
from tensorly_custom.base import unfold

def unfold_from_vectorized_sparse(
    vec_tensor: cpx_sparse.spmatrix,
    orig_shape,
    mode: int,
    to_dense: bool = False,
):
    """
    Unfold a sparse tensor that is stored as a vectorized CuPy sparse matrix.

    Parameters
    ----------
    vec_tensor : cupyx.scipy.sparse.spmatrix
        Sparse matrix of shape (np.prod(orig_shape), 1) created by
        `torch_sparse_to_cupy` for an N-D tensor.
    orig_shape : tuple[int, ...]
        Original N-D tensor shape, e.g. (I0, I1, I2).
    mode : int
        Mode along which to unfold.
    to_dense : bool, default False
        If True, return a dense cupy.ndarray.
        If False, return a cupy sparse COO matrix.

    Returns
    -------
    unfolded : cupy.ndarray or cupyx.scipy.sparse.coo_matrix
        Mode-`mode` unfolding of shape
        (orig_shape[mode], np.prod(orig_shape) // orig_shape[mode]).
    """

    cu = vec_tensor.tocoo()

    row_cp = cu.row
    col_cp = cu.col
    data_cp = cu.data

    orig_shape = tuple(orig_shape)
    size = int(np.prod(orig_shape))
    int32_max = np.iinfo(np.int32).max
    block_size = min(size, int32_max)

    # ---- move to host and use int64 for safe arithmetic ----
    row_np = cp.asnumpy(row_cp).astype(np.int64)
    col_np = cp.asnumpy(col_cp).astype(np.int64)

    flat_np = row_np + col_np * np.int64(block_size)

    coords = np.unravel_index(flat_np, orig_shape)

    row_unf_np = coords[mode]

    other_coords = coords[:mode] + coords[mode + 1 :]
    other_shape = tuple(s for i, s in enumerate(orig_shape) if i != mode)
    col_unf_np = np.ravel_multi_index(other_coords, other_shape)

    row_unf_cp = cp.asarray(row_unf_np)
    col_unf_cp = cp.asarray(col_unf_np)

    unfolded_shape = (orig_shape[mode], int(np.prod(other_shape)))
    unfolded = cpx_sparse.coo_matrix(
        (data_cp, (row_unf_cp, col_unf_cp)),
        shape=unfolded_shape,
    )

    if to_dense:
        return unfolded.toarray()
    return unfolded
factors_copy = [f.copy() for f in factors]


In [11]:
# we copy factors to efficiently compare the approaches
for mode in modes:
    # the first steps can be done with the dense implementation
    B = tucker_to_tensor((core, factors), skip_factor=mode)
    B = tl.transpose(unfold(B, mode))
    unfolded = unfold_from_vectorized_sparse(tensor, shape, mode) # this is the same as B when using dense tensor!
    numerator = unfolded @ B  # cupy sparse @ dense
    numerator = tl.clip(numerator, a_min=epsilon, a_max=None)
    denominator = tl.dot(factors[mode], tl.dot(tl.transpose(B), B))
    denominator = tl.clip(denominator, a_min=epsilon, a_max=None)
    factors[mode] *= numerator / denominator

# this multiplies gpu usage, but we still remain in **very** efficient territory

In [10]:
# alternative:
from tensorly_custom.tenalg import mode_dot
for mode in modes:
    # 1. compressed data
    modes_others = [m for m in modes if m != mode]
    Y = sparse_multi_mode_dot_vec(
        vec_tensor=tensor,
        orig_shape=shape,
        factors=factors_copy,
        modes=modes_others,
        transpose_factors=True,
    )  # shape with dim[mode] = I_mode, others R_k

    Y_perm = cp.moveaxis(Y, mode, 0)
    I_mode = shape[mode]
    R_rest = int(cp.prod(cp.array(Y_perm.shape[1:])))
    Y_n = Y_perm.reshape(I_mode, R_rest)    # (I_mode, ∏_{k≠mode} R_k)

    # 2. build \tilde G via Gram matrices (same grams as before)
    tilde_core = core
    for k in modes_others:
        G_k = tl.dot(tl.transpose(factors_copy[k]), factors_copy[k])  # (R_k, R_k)
        tilde_core = mode_dot(tilde_core, G_k, k)

    # 3. unfold \tilde G along mode n
    tilde_G_n = unfold(tilde_core, mode)         # (R_mode, R_rest)
    M = tl.transpose(tilde_G_n)                  # (R_rest, R_mode)

    # 4. compute numerator / denominator in compressed space
    numerator = Y_n @ M                          # (I_mode, R_mode)
    numerator = tl.clip(numerator, a_min=epsilon, a_max=None)

    K_n = tl.dot(tilde_G_n, tl.transpose(tilde_G_n))  # (R_mode, R_mode)
    denominator = tl.dot(factors_copy[mode], K_n)
    denominator = tl.clip(denominator, a_min=epsilon, a_max=None)

    # 5. multiplicative update
    factors_copy[mode] *= numerator / denominator


Applying mode 1
Applying mode 2
Final core shape (sparse): (5000, 50, 50)
Applying mode 0
Applying mode 2
Final core shape (sparse): (50, 5000, 50)
Applying mode 0
Applying mode 1
Final core shape (sparse): (50, 50, 5000)


In [13]:
for mode in modes:
    # shared starting point
    base_factors = [f.copy() for f in factors]
    core_base = core.copy()  # or tl.copy(core) if needed

    # --- baseline (dense-B) update just for this mode ---
    factors_A = [f.copy() for f in base_factors]
    B = tucker_to_tensor((core_base, factors_A), skip_factor=mode)
    B = tl.transpose(unfold(B, mode))
    unfolded = unfold_from_vectorized_sparse(tensor, shape, mode)

    num_A = unfolded @ B
    num_A = tl.clip(num_A, a_min=epsilon, a_max=None)
    den_A = tl.dot(factors_A[mode], tl.dot(tl.transpose(B), B))
    den_A = tl.clip(den_A, a_min=epsilon, a_max=None)
    updated_A = factors_A[mode] * (num_A / den_A)

    # --- compressed update just for this mode ---
    factors_B = [f.copy() for f in base_factors]

    modes_others = [m for m in modes if m != mode]
    Y = sparse_multi_mode_dot_vec(
        vec_tensor=tensor,
        orig_shape=shape,
        factors=factors_B,
        modes=modes_others,
        transpose_factors=True,
    )

    Y_perm = cp.moveaxis(Y, mode, 0)
    I_mode = shape[mode]
    R_rest = int(cp.prod(cp.array(Y_perm.shape[1:])))
    Y_n = Y_perm.reshape(I_mode, R_rest)

    tilde_core = core_base.copy()
    for k in modes_others:
        G_k = tl.dot(tl.transpose(factors_B[k]), factors_B[k])
        tilde_core = mode_dot(tilde_core, G_k, k)

    tilde_G_n = unfold(tilde_core, mode)
    M = tl.transpose(tilde_G_n)

    num_B = Y_n @ M
    num_B = tl.clip(num_B, a_min=epsilon, a_max=None)

    K_n = tl.dot(tilde_G_n, tl.transpose(tilde_G_n))
    den_B = tl.dot(factors_B[mode], K_n)
    den_B = tl.clip(den_B, a_min=epsilon, a_max=None)

    updated_B = factors_B[mode] * (num_B / den_B)

    diff = tl.norm(updated_A - updated_B)
    print(f"Mode {mode}: update difference norm = {diff}")


Applying mode 1
Applying mode 2
Final core shape (sparse): (5000, 50, 50)
Mode 0: update difference norm = 8.443707599781192e-08
Applying mode 0
Applying mode 2
Final core shape (sparse): (50, 5000, 50)
Mode 1: update difference norm = 29.214008331298828
Applying mode 0
Applying mode 1
Final core shape (sparse): (50, 50, 5000)
Mode 2: update difference norm = 15.40970230102539


## 2.2 core update
Here comes the hard part:
tucker_to_tensor implies recombining the original tensor and the new factors


In [54]:
# numerator = tucker_to_tensor((tensor, factors), transpose_factors=True)
skip_factor=None
transpose_factors=False
# modes=None
"""goal:
numerator = multi_mode_dot(
        tensor, factors, skip=skip_factor, transpose=transpose_factors, modes=modes
    )
"""

'goal:\nnumerator = multi_mode_dot(\n        tensor, factors, skip=skip_factor, transpose=transpose_factors, modes=modes\n    )\n'

In [55]:
A = tl.transpose(factors[0])
folded_A_tensor = unfold_from_vectorized_sparse(tensor, shape, 0, to_dense=False)
print('A shape', A.shape)
print('tensor shape', tensor.shape)
print('folded A shape', folded_A_tensor.shape)
dot = A @ folded_A_tensor  # cupy sparse @ dense
print('dot shape', dot.shape)
print(type(dot))

A shape (50, 1000)
tensor shape (1000000000, 1)
folded A shape (1000, 1000000)
dot shape (50, 1000000)
<class 'cupy.ndarray'>


In [56]:
# def left_dense_mul_sparse(mat: cp.ndarray, sp: cpx_sparse.spmatrix) -> cpx_sparse.coo_matrix:
#     """
#     Compute mat @ sp but keep the result sparse (COO).
#
#     mat: cupy ndarray of shape (R, I_mode)
#     sp:  cupy sparse matrix (COO/CSR/CSC) of shape (I_mode, K)
#
#     Returns
#     -------
#     res : cupyx.scipy.sparse.coo_matrix of shape (R, K)
#     """
#     sp = sp.tocoo()
#     row = sp.row      # (nnz,)
#     col = sp.col      # (nnz,)
#     data = sp.data    # (nnz,)
#
#     R = mat.shape[0]
#     I_mode = mat.shape[1]
#     assert I_mode == sp.shape[0], f"mat shape {mat.shape} not compatible with sparse {sp.shape}"
#
#     # Gather the relevant columns of mat for each nonzero row index
#     # mat_cols: shape (R, nnz), column j is mat[:, row[j]]
#     mat_cols = mat[:, row]  # dense
#
#     # Scale each column by the nonzero value
#     # data has shape (nnz,), broadcasting to (R, nnz)
#     vals = mat_cols * data
#
#     nnz = data.shape[0]
#
#     # Build indices for the output COO
#     # For each original nonzero (row[j], col[j]) we now have R contributions,
#     # one for each output row r in [0, R)
#     row_out = cp.repeat(cp.arange(R), nnz)   # (R*nnz,)
#     col_out = cp.tile(col, R)               # (R*nnz,)
#     data_out = vals.reshape(-1)             # (R*nnz,)
#
#     res = cpx_sparse.coo_matrix((data_out, (row_out, col_out)),
#                                 shape=(R, sp.shape[1]))
#     # Combine duplicates if any
#     res.sum_duplicates()
#     return res


In [59]:
# tensor is your vectorized CuPy sparse matrix (prod(shape), 1)
numerator = sparse_multi_mode_dot_vec(
    vec_tensor=tensor,
    orig_shape=shape,
    factors=factors,
    modes=modes,
    transpose_factors=True,  # X ×_n W_n^T
)
# we clip the numerator
numerator = tl.clip(numerator, a_min=epsilon, a_max=None)
print("core_numer shape:", numerator.shape)  # should be (50, 50, 50)
# only 300 mb large!

Applying mode 0
Applying mode 1
Applying mode 2
Final core shape (sparse): (50, 50, 50)
core_numer shape: (50, 50, 50)


In [60]:
from tensorly_custom.tenalg import mode_dot
# these operations can again be done with the dense implementation
for i, f in enumerate(factors):
    if i:
        denominator = mode_dot(denominator, tl.dot(tl.transpose(f), f), i)
    else:
        denominator = mode_dot(core, tl.dot(tl.transpose(f), f), i)
denominator = tl.clip(denominator, a_min=epsilon, a_max=None)


In [61]:
core *= numerator / denominator

## We couldn't do it, so below is implemented by GPT

In [62]:
norm_X_sq = norm_tensor ** 2
norm_Xhat_sq = tl.sum(core * denominator)
inner_prod = tl.sum(numerator * core)
residual_norm = tl.sqrt(norm_X_sq + norm_Xhat_sq - 2 * inner_prod)
relative_error = residual_norm / norm_tensor
relative_error

array(0.71388006, dtype=float32)

# Old code

In [ ]:
# Exploding GPU implementation
# we encode the core as a sparse vector
core_vector = tl.reshape(core, (-1, 1))
core_vector_cp = cp.asarray(core_vector)
core_sparse = cpx_sparse.coo_matrix(
    (core_vector_cp.flatten(), (cp.arange(core_vector_cp.shape[0]), cp.zeros(core_vector_cp.shape[0]))),
    shape=(core_vector_cp.shape[0], 1),
)
# reconstruction = sparse_multi_mode_dot_vec(
#     vec_tensor=core_sparse,
#     orig_shape=rank,
#     factors=factors,
#     modes=modes,
#     transpose_factors=False,  # reconstructing: core ×_n W_n
# )
# print("reconstruction shape:", reconstruction.shape)  # should be (prod(shape), 1

array(0.99995476, dtype=float32)

## Old, working but large GPU explosions


In [21]:
def fold_unfolded_sparse_to_vec(
    unfolded: cpx_sparse.spmatrix,
    old_shape: Tuple[int, ...],
    mode: int,
    new_dim: int,
) -> Tuple[cpx_sparse.coo_matrix, Tuple[int, ...]]:
    """
    Fold a mode-`mode` unfolded sparse matrix back to a vectorized sparse tensor.

    unfolded: sparse COO of shape (new_dim, prod(other_dims))
    old_shape: original shape BEFORE replacing dimension at `mode`
    mode: mode index that was unfolded
    new_dim: new size at `mode` (typically rank[mode])

    Returns
    -------
    vec_sparse: COO of shape (prod(new_shape), 1)
    new_shape:  tuple of ints, updated tensor shape
    """
    unfolded = unfolded.tocoo()
    row = unfolded.row     # (nnz,)
    col = unfolded.col     # (nnz,)
    data = unfolded.data   # (nnz,)

    old_shape = tuple(old_shape)
    N = len(old_shape)

    # Build new shape
    new_shape = list(old_shape)
    new_shape[mode] = new_dim
    new_shape = tuple(new_shape)

    # Other dimensions (all except mode)
    other_shape = tuple(s for i, s in enumerate(old_shape) if i != mode)

    # Decode column index into coordinates of the "other" modes
    coords_other = cp.unravel_index(col, other_shape)  # tuple of (N-1) arrays

    # Now assemble full coordinates (length N) in the new tensor:
    # at position `mode` we put row, others we take from coords_other
    coords_full = []
    idx_other = 0
    for i in range(N):
        if i == mode:
            coords_full.append(row)
        else:
            coords_full.append(coords_other[idx_other])
            idx_other += 1

    coords_full = tuple(coords_full)
    # Flatten multi-index into a single index for the vectorized representation
    flat = cp.ravel_multi_index(coords_full, new_shape)  # (nnz,)
    zero = cp.zeros_like(flat)  # column index always 0 for (prod(new_shape), 1)
    vec_sparse = cpx_sparse.coo_matrix(
        (data, (flat, zero)),
        shape=(int(np.prod(new_shape)), 1),
    )
    vec_sparse.sum_duplicates()
    return vec_sparse, new_shape

def sparse_mode_dot_vec(
    vec_tensor: cpx_sparse.spmatrix,
    curr_shape: Tuple[int, ...],
    factor: cp.ndarray,
    mode: int,
    transpose_factor: bool = True,
) -> Tuple[cpx_sparse.coo_matrix, Tuple[int, ...]]:
    """
    Perform a mode-`mode` product on a vectorized sparse tensor (prod(curr_shape), 1),
    using a dense factor matrix, and return the new vectorized sparse tensor.

    vec_tensor: sparse COO (prod(curr_shape), 1)
    curr_shape: current tensor shape
    factor:     dense matrix of shape (I_mode, R_mode) (or R_mode, I_mode if transpose_factor=False)
    mode:       mode index in [0, len(curr_shape))
    transpose_factor: if True, use factor.T (for Tucker-style X ×_n W_n^T)

    Returns
    -------
    new_vec:   sparse COO (prod(new_shape), 1)
    new_shape: updated shape, with dimension at `mode` replaced by R_mode
    """
    curr_shape = tuple(curr_shape)
    I_mode = curr_shape[mode]

    # Factor handling
    if transpose_factor:
        # factor is (I_mode, R_mode) => mat is (R_mode, I_mode)
        assert factor.shape[0] == I_mode, f"factor shape {factor.shape} not compatible with dim {I_mode}"
        mat = tl.transpose(factor)  # (R_mode, I_mode)
    else:
        # factor is already (R_mode, I_mode)
        assert factor.shape[1] == I_mode, f"factor shape {factor.shape} not compatible with dim {I_mode}"
        mat = factor

    R_mode = mat.shape[0]
    # 1) Unfold current sparse tensor along this mode (sparse COO)
    unfolded = unfold_from_vectorized_sparse(
        vec_tensor,
        curr_shape,
        mode,
        to_dense=False,
    )  # shape: (I_mode, prod(other_dims))
    # 2) Left-multiply with dense matrix, but keep result sparse
    #    -> shape: (R_mode, prod(other_dims))
    unfolded_new = left_dense_mul_sparse(mat, unfolded)
    # 3) Fold back into a new vectorized sparse tensor with updated shape
    new_vec, new_shape = fold_unfolded_sparse_to_vec(
        unfolded_new,
        old_shape=curr_shape,
        mode=mode,
        new_dim=R_mode,
    )
    return new_vec, new_shape
def sparse_multi_mode_dot_vec(
    vec_tensor: cpx_sparse.spmatrix,
    orig_shape: Tuple[int, ...],
    factors: List[cp.ndarray],
    modes: Optional[List[int]] = None,
    transpose_factors: bool = True,
) -> cp.ndarray:
    """
    multi_mode_dot for a vectorized sparse tensor (prod(orig_shape), 1),
    applying dense factor matrices along the given modes, **staying sparse**
    until the final (small) result, which is densified.

    vec_tensor: sparse COO (prod(orig_shape), 1)
    orig_shape: original tensor shape
    factors:    list of factor matrices, one per mode index
                factor[n] has shape (I_n, R_n)
    modes:      list of modes to apply; if None, uses range(len(factors))
    transpose_factors: if True, uses factors[n].T (Tucker-style)
    """
    if modes is None:
        modes = list(range(len(factors)))

    current_vec = vec_tensor
    current_shape = tuple(orig_shape)

    # Apply each mode in any order (commutes)
    for mode in modes:
        print("Applying mode", mode)
        current_vec, current_shape = sparse_mode_dot_vec(
            current_vec,
            current_shape,
            factors[mode],
            mode=mode,
            transpose_factor=transpose_factors,
        )

    # At this point, current_vec is still sparse (prod(core_shape), 1)
    core_shape = current_shape  # typically your (50, 50, 50) or similar
    print("Final core shape (sparse):", core_shape)
    # Finally densify the small core
    coo = current_vec.tocoo()
    flat = coo.row
    data = coo.data

    # Build dense core
    coords = cp.unravel_index(flat, core_shape)
    core_dense = cp.zeros(core_shape, dtype=data.dtype)
    core_dense[coords] = data

    return core_dense


In [22]:
# tensor is your vectorized CuPy sparse matrix (prod(shape), 1)
numerator = sparse_multi_mode_dot_vec(
    vec_tensor=tensor,
    orig_shape=shape,
    factors=factors,
    modes=modes,
    transpose_factors=True,  # X ×_n W_n^T
)

print("core_numer shape:", numerator.shape)  # should be (50, 50, 50)

Applying mode 0


AttributeError: 'ndarray' object has no attribute 'tocoo'

In [15]:
# we clip this core
numerator = tl.clip(numerator, a_min=epsilon, a_max=None)

In [16]:
from tensorly_custom.tenalg import mode_dot
# these operations can again be done with the dense implementation
for i, f in enumerate(factors):
    if i:
        denominator = mode_dot(denominator, tl.dot(tl.transpose(f), f), i)
    else:
        denominator = mode_dot(core, tl.dot(tl.transpose(f), f), i)

In [17]:
denominator = tl.clip(denominator, a_min=epsilon, a_max=None)


In [18]:
core *= numerator / denominator

In [19]:
reconstruction = tucker_to_tensor((core, factors))
# we flatten this to compute the error
flattened_reconstruction = tl.reshape(reconstruction, (-1, 1))
rec_error = (
            tl.norm(tensor - flattened_reconstruction, 2) / norm_tensor
        )

tucker_to_tensor time: 0.0018360614776611328


In [26]:
rec_error

array(0.99995565, dtype=float32)